In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import random
import copy

import numpy as np
import pandas as pd
import torch
from torch import nn

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    recall_score,
)

BASE = Path("/content/drive/MyDrive/tomato_lab2field")
FEATURE_DIR = BASE / "artifacts" / "dinov2_raw"
RESULT_DIR = BASE / "results" / "dinov2_raw_clean"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Mounted at /content/drive
Device: cpu


In [2]:
cls_features = torch.load(
    FEATURE_DIR / "cls_features.pt",
    map_location="cpu"
).float()

gap_features = torch.load(
    FEATURE_DIR / "gap_features.pt",
    map_location="cpu"
).float()

meta = pd.read_csv(FEATURE_DIR / "metadata.csv")

print("CLS:", cls_features.shape)
print("GAP:", gap_features.shape)
print("Metadata:", len(meta))

CLS: torch.Size([8735, 768])
GAP: torch.Size([8735, 768])
Metadata: 8735


In [3]:
assert cls_features.shape == (8735, 768)
assert gap_features.shape == (8735, 768)
assert len(meta) == 8735

assert (meta["split"] == "train").sum() == 5692
assert (meta["split"] == "validation").sum() == 1221
assert (meta["split"] == "test").sum() == 1219
assert (meta["split"] == "external_target_test").sum() == 603

assert meta["canonical_label"].nunique() == 6

print("Data contract checks passed.")
print(meta["split"].value_counts())

Data contract checks passed.
split
train                   5692
validation              1221
test                    1219
external_target_test     603
Name: count, dtype: int64


In [4]:
SEEDS = [42, 0, 1]

CONFIG = {
    "backbone": "Frozen DINOv2 ViT-B/14",
    "probe": "Linear(768 -> 6)",
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 50,
    "patience": 5,
    "batch_size": 256,
    "selection_metric": "PlantVillage validation Macro-F1",
}

with open(RESULT_DIR / "probe_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

class_names = sorted(meta["canonical_label"].unique())
num_classes = len(class_names)

labels = torch.tensor(meta["label"].values, dtype=torch.long)

split_idx = {
    "train": np.where(meta["split"].values == "train")[0],
    "validation": np.where(meta["split"].values == "validation")[0],
    "test": np.where(meta["split"].values == "test")[0],
    "target": np.where(
        meta["split"].values == "external_target_test"
    )[0],
}

print(class_names)

['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_mosaic_virus']


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def evaluate(model, X, y):
    model.eval()

    with torch.no_grad():
        logits = model(X.to(device))
        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

    y_np = y.numpy()
    p_np = preds.numpy()

    metrics = {
        "macro_f1": float(
            f1_score(y_np, p_np, average="macro")
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_np, p_np)
        ),
        "accuracy": float(
            accuracy_score(y_np, p_np)
        ),
        "per_class_recall": recall_score(
            y_np,
            p_np,
            labels=list(range(num_classes)),
            average=None,
            zero_division=0,
        ).tolist(),
    }

    return metrics, preds, probs


def save_predictions(path, subset_meta, y, preds, probs):
    out = subset_meta.copy().reset_index(drop=True)

    out["prediction"] = [
        class_names[i] for i in preds.numpy()
    ]

    out["confidence"] = probs.max(dim=1).values.numpy()

    out["correct"] = (
        preds.numpy() == y.numpy()
    )

    out[
        [
            "image_id",
            "canonical_label",
            "prediction",
            "confidence",
            "correct",
        ]
    ].to_csv(path, index=False)

In [6]:
def train_probe(features, representation, seed):
    set_seed(seed)

    output_dir = RESULT_DIR / representation / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_idx = split_idx["train"]
    val_idx = split_idx["validation"]
    test_idx = split_idx["test"]
    target_idx = split_idx["target"]

    X_train = features[train_idx]
    y_train = labels[train_idx]

    X_val = features[val_idx]
    y_val = labels[val_idx]

    X_test = features[test_idx]
    y_test = labels[test_idx]

    X_target = features[target_idx]
    y_target = labels[target_idx]

    model = nn.Linear(768, num_classes).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    criterion = nn.CrossEntropyLoss()

    generator = torch.Generator().manual_seed(seed)

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train, y_train),
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        generator=generator,
    )

    best_f1 = -1.0
    best_epoch = -1
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(1, CONFIG["max_epochs"] + 1):
        model.train()

        total_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)

        train_loss = total_loss / len(X_train)

        val_metrics, _, _ = evaluate(
            model,
            X_val,
            y_val,
        )

        val_f1 = val_metrics["macro_f1"]

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_macro_f1": val_f1,
        })

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CONFIG["patience"]:
            break

    model.load_state_dict(best_state)

    pv_metrics, pv_preds, pv_probs = evaluate(
        model,
        X_test,
        y_test,
    )

    pd_metrics, pd_preds, pd_probs = evaluate(
        model,
        X_target,
        y_target,
    )

    result = {
        "representation": representation,
        "seed": seed,
        "best_epoch": best_epoch,
        "validation_macro_f1": best_f1,

        "pv_macro_f1": pv_metrics["macro_f1"],
        "pv_balanced_accuracy": pv_metrics["balanced_accuracy"],
        "pv_accuracy": pv_metrics["accuracy"],

        "pd_macro_f1": pd_metrics["macro_f1"],
        "pd_balanced_accuracy": pd_metrics["balanced_accuracy"],
        "pd_accuracy": pd_metrics["accuracy"],

        "delta_macro_f1":
            pv_metrics["macro_f1"] - pd_metrics["macro_f1"],

        "pd_per_class_recall": {
            class_names[i]: pd_metrics["per_class_recall"][i]
            for i in range(num_classes)
        },
    }

    with open(output_dir / "metrics.json", "w") as f:
        json.dump(result, f, indent=2)

    pd.DataFrame(history).to_csv(
        output_dir / "training_history.csv",
        index=False,
    )

    save_predictions(
        output_dir / "predictions_pv_test.csv",
        meta.iloc[test_idx],
        y_test,
        pv_preds,
        pv_probs,
    )

    save_predictions(
        output_dir / "predictions_plantdoc.csv",
        meta.iloc[target_idx],
        y_target,
        pd_preds,
        pd_probs,
    )

    torch.save(
        {
            "state_dict": best_state,
            "representation": representation,
            "seed": seed,
            "best_epoch": best_epoch,
            "best_validation_macro_f1": best_f1,
            "class_names": class_names,
            "config": CONFIG,
        },
        output_dir / "best_probe.pt",
    )

    return result

In [8]:
results = []

for representation, features in [
    ("cls", cls_features),
    ("gap", gap_features),
]:

    print(f"\n{representation.upper()}")

    for seed in SEEDS:
        r = train_probe(features, representation, seed)
        results.append(r)

        print(
            f"seed={seed} | "
            f"PV F1={r['pv_macro_f1']:.4f} | "
            f"PD F1={r['pd_macro_f1']:.4f} | "
            f"Δ={r['delta_macro_f1']:.4f}"
        )


CLS
seed=42 | PV F1=0.9804 | PD F1=0.3121 | Δ=0.6683
seed=0 | PV F1=0.9793 | PD F1=0.2973 | Δ=0.6821
seed=1 | PV F1=0.9768 | PD F1=0.2701 | Δ=0.7067

GAP
seed=42 | PV F1=0.9812 | PD F1=0.2483 | Δ=0.7329
seed=0 | PV F1=0.9836 | PD F1=0.2452 | Δ=0.7384
seed=1 | PV F1=0.9700 | PD F1=0.2527 | Δ=0.7172


In [9]:
summary = pd.DataFrame([
    {
        "representation": r["representation"],
        "seed": r["seed"],
        "best_epoch": r["best_epoch"],
        "pv_macro_f1": r["pv_macro_f1"],
        "pv_balanced_accuracy": r["pv_balanced_accuracy"],
        "pv_accuracy": r["pv_accuracy"],
        "pd_macro_f1": r["pd_macro_f1"],
        "pd_balanced_accuracy": r["pd_balanced_accuracy"],
        "pd_accuracy": r["pd_accuracy"],
        "delta_macro_f1": r["delta_macro_f1"],
    }
    for r in results
])

summary.to_csv(
    RESULT_DIR / "seed_results.csv",
    index=False,
)

print(summary)

print("\nMEAN ± STD")

for representation in ["cls", "gap"]:
    sub = summary[
        summary["representation"] == representation
    ]

    print(f"\n{representation.upper()}")

    for col in [
        "pv_macro_f1",
        "pv_balanced_accuracy",
        "pv_accuracy",
        "pd_macro_f1",
        "pd_balanced_accuracy",
        "pd_accuracy",
        "delta_macro_f1",
    ]:
        print(
            f"{col}: "
            f"{sub[col].mean():.5f} ± "
            f"{sub[col].std(ddof=1):.5f}"
        )

  representation  seed  best_epoch  pv_macro_f1  pv_balanced_accuracy  \
0            cls    42          33     0.980430              0.977580   
1            cls     0          19     0.979348              0.976957   
2            cls     1          19     0.976786              0.974140   
3            gap    42          26     0.981231              0.977726   
4            gap     0          31     0.983555              0.981714   
5            gap     1          18     0.969960              0.964133   

   pv_accuracy  pd_macro_f1  pd_balanced_accuracy  pd_accuracy  delta_macro_f1  
0     0.978671     0.312089              0.364733     0.402985        0.668342  
1     0.977030     0.297293              0.349163     0.386401        0.682055  
2     0.976210     0.270077              0.322559     0.349917        0.706710  
3     0.982773     0.248299              0.313368     0.364842        0.732932  
4     0.984413     0.245198              0.305672     0.351575        0.738357  
5 